In [ ]:
import time
from datetime import datetime
import sqlite3
import requests
import pandas as pd

# 讀取目標停車場列表
df = pd.read_csv("carparks_kwuntong_shatin.csv")
target_ids = set(df["park_id"].astype(str))

def collect_vacancy():
    url = "https://api.data.gov.hk/v1/carpark-info-vacancy"
    params = {
        "data": "vacancy",
        "lang": "zh_TW",
        "vehicleTypes": "privateCar"
    }
    
    response = requests.get(url, params=params, timeout=30)
    if response.status_code != 200:
        print("請求失敗")
        return
    
    vacancy_results = response.json().get("results", [])
    records = []
    for item in vacancy_results:
        park_id = str(item.get("park_Id"))
        if park_id not in target_ids:
            continue
        private = item.get("privateCar", [])
        if private:
            v = private[0]
            records.append({
                "park_id": park_id,
                "vacancy": v.get("vacancy"),
                "vacancy_type": v.get("vacancy_type"),
                "lastupdate": v.get("lastupdate")
            })
    
    if not records:
        print("沒有新資料")
        return
    
    collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect("parking.db")
    cursor = conn.cursor()
    
    for r in records:
        cursor.execute("""
            INSERT INTO vacancy_records (park_id, vacancy, vacancy_type, lastupdate, collected_at)
            VALUES (?, ?, ?, ?, ?)
        """, (r["park_id"], r["vacancy"], r["vacancy_type"], r["lastupdate"], collected_at))
    
    conn.commit()
    conn.close()
    print(f"成功收集並寫入 {len(records)} 筆資料 ({collected_at})")
    
def collect_weather():
    weather_url = "https://data.weather.gov.hk/weatherAPI/opendata/weather.php"
    try:
        resp = requests.get(weather_url, params={"dataType": "rhrread", "lang": "en"}, timeout=30)
        weather = resp.json()
        
        hko_temp = None
        for t in weather.get("temperature", {}).get("data", []):
            if "Observatory" in t.get("place", ""):
                hko_temp = t.get("value")
                break
        
        rainfall_list = weather.get("rainfall", {}).get("data", [])
        max_rain = max([r.get("max", 0) for r in rainfall_list]) if rainfall_list else 0
        
        collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        conn = sqlite3.connect("parking.db")
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO weather_records (collected_at, temperature, max_rainfall, update_time)
            VALUES (?, ?, ?, ?)
        """, (collected_at, hko_temp, max_rain, weather.get("updateTime")))
        conn.commit()
        conn.close()
        print(f"天氣數據已收集 ({collected_at})")
    except Exception as e:
        print("天氣收集失敗:", e)

if __name__ == "__main__":
    print("開始持續收集（每 10 分鐘一次），按 Ctrl+C 停止")
    while True:
        collect_vacancy()
        collect_weather()
        time.sleep(600)  # 600 秒 = 10 分鐘

開始持續收集（每 10 分鐘一次），按 Ctrl+C 停止
成功收集並寫入 149 筆資料 (2026-10-19 18:10:32)
天氣數據已收集 (2026-10-19 18:10:32)
成功收集並寫入 149 筆資料 (2026-10-19 18:20:34)
天氣數據已收集 (2026-10-19 18:20:34)
成功收集並寫入 149 筆資料 (2026-10-19 18:30:37)
天氣數據已收集 (2026-10-19 18:30:37)
成功收集並寫入 149 筆資料 (2026-10-19 18:40:37)
天氣數據已收集 (2026-10-19 18:40:38)
